In [75]:
import pandas as pd
import numpy as np
import random
import copy
import os
import json
import time

# ==========================================
# PARAMETER DPSO & CONSTRAINT VRP
# ==========================================
SWARM_SIZE = 60
MAX_ITER = 500
PATIENCE = 100

P_COG = 0.6
P_SOC = 0.3
P_INERTIA = 0.8

# Constraint Waktu (dalam satuan menit)
SERVICE_TIME = 20          # 20 menit service hour per puskesmas
WORK_HOUR_LIMIT = 600      # 10 jam kerja = 600 menit

SEED_BASE = 42
TOTAL_RUN = 10

DATA_DIR = "../data"
OUTPUT_DIR = "../output_json"
path_koordinat = os.path.join(DATA_DIR, "koordinat_eas.csv")

print("✅ Libraries and parameters loaded successfully!")

✅ Libraries and parameters loaded successfully!


In [76]:
def clean_coordinate(val, is_lat=True):
    """Pembersih otomatis koordinat rusak akibat format regional CSV Excel"""
    val_str = str(val).strip().replace('"', '').replace("'", "")
    digits = val_str.replace(',', '').replace('.', '').replace('-', '')
    if not digits:
        return 0.0
    
    is_negative = val_str.startswith('-')
    
    if is_lat:
        cleaned = f"-{digits[0]}.{digits[1:]}" if is_negative else f"{digits[0]}.{digits[1:]}"
    else:
        cleaned = f"{digits[:3]}.{digits[3:]}"
    return float(cleaned)

def get_priority_score(jaringan, jenis):
    """Mapping prioritas sesuai dataset"""
    jaringan_lower = str(jaringan).lower()
    jenis_lower = str(jenis).lower()
    
    if "induk" in jaringan_lower and "inap" in jenis_lower:
        return 1
    elif "induk" in jaringan_lower and "jalan" in jenis_lower:
        return 2
    else:
        return 3

print("✅ Coordinate cleaner and priority function defined!")

✅ Coordinate cleaner and priority function defined!


In [77]:
def decode_and_evaluate(permutation, dist_matrix, time_matrix, lokasi_list, priority_map):
    routes = []
    current_route = []
    current_time = 0.0
    current_dist = 0.0
    
    for node in permutation:
        if not current_route:
            time_needed = time_matrix[0][node] + SERVICE_TIME + time_matrix[node][0]
            current_route.append(node)
            current_time = time_matrix[0][node] + SERVICE_TIME
            current_dist = dist_matrix[0][node]
        else:
            last_node = current_route[-1]
            time_to_next = time_matrix[last_node][node] + SERVICE_TIME
            time_return_from_next = time_matrix[node][0]
            
            if current_time + time_to_next + time_return_from_next <= WORK_HOUR_LIMIT:
                current_route.append(node)
                current_time += time_to_next
                current_dist += dist_matrix[last_node][node]
            else:
                current_time += time_matrix[last_node][0]
                current_dist += dist_matrix[last_node][0]
                routes.append({"nodes": current_route, "waktu": current_time, "jarak": current_dist})
                
                current_route = [node]
                current_time = time_matrix[0][node] + SERVICE_TIME
                current_dist = dist_matrix[0][node]
                
    if current_route:
        current_time += time_matrix[current_route[-1]][0]
        current_dist += dist_matrix[current_route[-1]][0]
        routes.append({"nodes": current_route, "waktu": current_time, "jarak": current_dist})
        
    total_actual_distance = sum(r["jarak"] for r in routes)
    total_actual_time = sum(r["waktu"] for r in routes)
    
    priority_penalty = 0.0
    for r in routes:
        for i in range(len(r["nodes"]) - 1):
            p1 = priority_map.get(lokasi_list[r["nodes"][i]], 3)
            p2 = priority_map.get(lokasi_list[r["nodes"][i+1]], 3)
            if p1 > p2:
                priority_penalty += 100.0
                
    fitness_value = total_actual_distance + priority_penalty
    return fitness_value, routes, total_actual_distance, total_actual_time

def get_swap_sequence(target, current):
    seq = []
    temp = current.copy()
    for i in range(len(target)):
        if temp[i] != target[i]:
            j = temp.index(target[i])
            seq.append((i, j))
            temp[i], temp[j] = temp[j], temp[i]
    return seq

def apply_velocity(route, velocity):
    new_route = route.copy()
    for i, j in velocity:
        new_route[i], new_route[j] = new_route[j], new_route[i]
    return new_route

print("✅ Decoder and DPSO operators compiled successfully!")

✅ Decoder and DPSO operators compiled successfully!


In [78]:
def run_dpso_vrp(dist_file, time_file, coord_df, priority_map):
    df_dist = pd.read_csv(dist_file, index_col=0)
    df_time = pd.read_csv(time_file, index_col=0)
    
    lokasi_list = [str(col).strip() for col in df_dist.columns]
    dist_matrix = df_dist.values
    time_matrix = df_time.values
    
    nodes = list(range(1, len(lokasi_list)))
    if len(nodes) == 0: return {}
        
    swarm = [random.sample(nodes, len(nodes)) for _ in range(SWARM_SIZE)]
    velocities = [[] for _ in range(SWARM_SIZE)]
    
    pbest = copy.deepcopy(swarm)
    pbest_fit = []
    for p in swarm:
        fit, _, _, _ = decode_and_evaluate(p, dist_matrix, time_matrix, lokasi_list, priority_map)
        pbest_fit.append(fit)
        
    best_idx = int(np.argmin(pbest_fit))
    gbest = copy.deepcopy(pbest[best_idx])
    gbest_fit, gbest_routes, gbest_dist, gbest_time = decode_and_evaluate(gbest, dist_matrix, time_matrix, lokasi_list, priority_map)
    
    vmax = int(3.0 * len(nodes))
    no_improve = 0
    riwayat_fitness = []
    
    for iterasi in range(MAX_ITER):
        improved = False
        for i in range(SWARM_SIZE):
            fit, _, _, _ = decode_and_evaluate(swarm[i], dist_matrix, time_matrix, lokasi_list, priority_map)
            if fit < pbest_fit[i]:
                pbest[i] = copy.deepcopy(swarm[i])
                pbest_fit[i] = fit
                
        best_idx = int(np.argmin(pbest_fit))
        if pbest_fit[best_idx] < gbest_fit:
            gbest = copy.deepcopy(pbest[best_idx])
            gbest_fit, gbest_routes, gbest_dist, gbest_time = decode_and_evaluate(gbest, dist_matrix, time_matrix, lokasi_list, priority_map)
            improved = True
            
        riwayat_fitness.append(round(gbest_dist, 2))
            
        if improved: no_improve = 0
        else: no_improve += 1
    
    
        # ===================================================================
        # PERBAIKAN LOGIKA VELOCITY & POSITION UPDATE (SEQUENTIAL DISCRETE)
        # ===================================================================
        for i in range(SWARM_SIZE):
            current_route = swarm[i].copy()
            new_v = []
            
            # 1. Komponen Inertia (Jalankan sisa velocity lama secara aman)
            for swap in velocities[i]:
                if random.random() < P_INERTIA:
                    if swap[0] < len(current_route) and swap[1] < len(current_route):
                        current_route[swap[0]], current_route[swap[1]] = current_route[swap[1]], current_route[swap[0]]
                        new_v.append(swap)
            
            # 2. Komponen Kognitif (Hitung ulang Swap Sequence dari posisi BARU ke PBest)
            v_cog = get_swap_sequence(pbest[i], current_route)
            for swap in v_cog:
                if random.random() < P_COG:
                    current_route[swap[0]], current_route[swap[1]] = current_route[swap[1]], current_route[swap[0]]
                    new_v.append(swap)
            
            # 3. Komponen Sosial (Hitung ulang Swap Sequence dari posisi BARU ke GBest)
            v_soc = get_swap_sequence(gbest, current_route)
            for swap in v_soc:
                if random.random() < P_SOC:
                    current_route[swap[0]], current_route[swap[1]] = current_route[swap[1]], current_route[swap[0]]
                    new_v.append(swap)
            
            if len(new_v) > vmax: 
                new_v = random.sample(new_v, vmax)
                
            velocities[i] = new_v
            swarm[i] = current_route
            
    rute_per_kurir_json = []
    for idx, r in enumerate(gbest_routes):
        urutan_nama = [lokasi_list[0]]
        koordinat_list = []
        
        depot_match = coord_df[coord_df['Nama Puskesmas'].str.strip() == lokasi_list[0].strip()]
        if not depot_match.empty:
            koordinat_list.append([
                clean_coordinate(depot_match.iloc[0]['Latitude'], is_lat=True),
                clean_coordinate(depot_match.iloc[0]['Longitude'], is_lat=False)
            ])
        else:
            koordinat_list.append([-7.32229, 112.77177])
            
        for n in r["nodes"]:
            nama_pusk = lokasi_list[n]
            urutan_nama.append(nama_pusk)
            pusk_match = coord_df[coord_df['Nama Puskesmas'].str.strip() == nama_pusk.strip()]
            if not pusk_match.empty:
                koordinat_list.append([
                    clean_coordinate(pusk_match.iloc[0]['Latitude'], is_lat=True),
                    clean_coordinate(pusk_match.iloc[0]['Longitude'], is_lat=False)
                ])
            else:
                koordinat_list.append([0.0, 0.0])
                
        urutan_nama.append(lokasi_list[0])
        koordinat_list.append(koordinat_list[0])
        
        rute_per_kurir_json.append({
            "id_kurir": idx + 1,
            "waktu_tempuh_menit": round(r["waktu"], 2),
            "jarak_tempuh_km": round(r["jarak"], 2),
            "urutan_kunjungan": urutan_nama,
            "koordinat_kunjungan": koordinat_list
        })
        
    return {
        "best_fitness": gbest_fit,
        "total_kurir": len(rute_per_kurir_json),
        "total_waktu_semua_menit": round(gbest_time, 2),
        "total_jarak_semua_km": round(gbest_dist, 2),
        "riwayat_konvergensi": riwayat_fitness,
        "rute_per_kurir": rute_per_kurir_json
    }

print("✅ Main DPSO VRP function compiled successfully with fix!")

✅ Main DPSO VRP function compiled successfully with fix!


In [79]:
if os.path.exists(path_koordinat):
    df_coords = pd.read_csv(path_koordinat)
    
    # Mapping prioritas
    priority_map = {str(row['Nama Puskesmas']).strip(): get_priority_score(row['Jaringan Pelayanan'], row['Jenis Layanan']) 
                    for _, row in df_coords.iterrows()}

    daftar_klaster = ["barat", "pusat", "selatan", "timur", "utara"]
    semua_file = os.listdir(DATA_DIR)

    output_gabungan = {
        "algoritma": "Discrete Particle Swarm Optimization (DPSO)",
        "hasil_per_klaster": {}
    }

    print(f"🚀 [START] Memulai optimasi batch ({TOTAL_RUN} run per klaster)...\n")

    for klaster in daftar_klaster:
        file_jarak_nama = next((f for f in semua_file if "jarak" in f.lower() and klaster in f.lower() and f.endswith('.csv')), None)
        file_waktu_nama = next((f for f in semua_file if "waktu" in f.lower() and klaster in f.lower() and f.endswith('.csv')), None)

        if file_jarak_nama and file_waktu_nama:
            print("="*70)
            print(f"📍 MEMPROSES KLASTER {klaster.upper()}")
            print("-" * 70)
            
            distance_history = []
            time_history = []
            best_fitness_overall = float('inf')
            best_run_data = None
            best_run_time = 0.0

            for run_idx in range(TOTAL_RUN):
                random.seed(SEED_BASE + run_idx)
                np.random.seed(SEED_BASE + run_idx)

                start_time = time.time()
                # Panggil fungsi optimasi
                hasil_klaster = run_dpso_vrp(os.path.join(DATA_DIR, file_jarak_nama), 
                                             os.path.join(DATA_DIR, file_waktu_nama), 
                                             df_coords, priority_map)
                end_time = time.time()
                
                waktu_detik = round(end_time - start_time, 3)
                fit_val = hasil_klaster["best_fitness"]
                dist_val = hasil_klaster["total_jarak_semua_km"]
                
                # --- TAMBAHAN OUTPUT RUN ---
                print(f"      ➔ Run {run_idx + 1}/{TOTAL_RUN} Selesai | Jarak: {round(dist_val, 2)} KM | Waktu: {waktu_detik} dtk")
                
                distance_history.append(fit_val)
                time_history.append(waktu_detik)
                
                import copy

                if fit_val < best_fitness_overall:
                    best_fitness_overall = fit_val
                    best_run_data = copy.deepcopy(hasil_klaster)
                    best_run_time = waktu_detik
                    best_convergence_history = copy.deepcopy(
                        hasil_klaster["riwayat_konvergensi"]
                    )
                    
                    
            arr_distance = np.array(distance_history)
            
            # Simpan ke dictionary
            klaster_key = klaster.capitalize()
            stats = {
                "fitness_minimum": round(float(np.min(arr_distance)), 2),
                "fitness_rata_rata": round(float(np.mean(arr_distance)), 2),
                "fitness_std_dev": round(float(np.std(arr_distance, ddof=1)), 2) if len(arr_distance) > 1 else 0.0,
                "waktu_komputasi_rata_rata_detik": round(float(np.mean(time_history)), 3),
                "semua_fitness_run": [round(float(d), 2) for d in distance_history]
            }
            
            output_gabungan["hasil_per_klaster"][klaster_key] = {
                "statistik_10_run": stats,
                "total_kurir": best_run_data["total_kurir"],
                "waktu_komputasi_detik_terbaik": round(best_run_time, 3),
                "total_waktu_semua_menit": best_run_data["total_waktu_semua_menit"],
                "total_jarak_semua_km": best_run_data["total_jarak_semua_km"],
                "riwayat_konvergensi": best_run_data["riwayat_konvergensi"],
                "rute_per_kurir": best_run_data["rute_per_kurir"]
            }

            # --- PREVIEW TERMINAL ---
            print(f"✅ HASIL TERBAIK KLASTER {klaster.upper()}:")
            print(f"Total Kurir          : {best_run_data['total_kurir']} Orang")
            print(f"Fitness Min (Jarak)  : {stats['fitness_minimum']} KM")
            print(f"Fitness Rata-Rata    : {stats['fitness_rata_rata']} KM")
            print(f"Standar Deviasi      : {stats['fitness_std_dev']}")
            print(f"Waktu Komputasi (Avg): {stats['waktu_komputasi_rata_rata_detik']} Detik")
            print("-" * 70)
            
            for kurir in best_run_data['rute_per_kurir']:
                print(f"🚚 [KURIR {kurir['id_kurir']}] - Jarak: {kurir['jarak_tempuh_km']} KM | Waktu: {kurir['waktu_tempuh_menit']} Menit")
                rute_singkat = " ➔ ".join([n.replace("Puskesmas ", "P. ").replace("Pustu ", "P. ") for n in kurir['urutan_kunjungan']])
                print(f"   Rute: {rute_singkat}\n")
        else:
            print(f"⚠️ Skip Klaster {klaster.upper()}: File tidak lengkap.")

    # --- EXPORT JSON ---
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    file_output = os.path.join(OUTPUT_DIR, "rute_dpso.json")
    with open(file_output, 'w') as f:
        json.dump(output_gabungan, f, indent=4)
    print("="*70)
    print(f"🎉 SEMUA KLASTER SELESAI! Hasil disimpan di: {file_output}")
else:
    print("❌ Gagal! File koordinat tidak ditemukan.")

🚀 [START] Memulai optimasi batch (10 run per klaster)...

📍 MEMPROSES KLASTER BARAT
----------------------------------------------------------------------
      ➔ Run 1/10 Selesai | Jarak: 185.35 KM | Waktu: 0.768 dtk
      ➔ Run 2/10 Selesai | Jarak: 194.01 KM | Waktu: 0.629 dtk
      ➔ Run 3/10 Selesai | Jarak: 236.39 KM | Waktu: 0.698 dtk
      ➔ Run 4/10 Selesai | Jarak: 197.22 KM | Waktu: 0.767 dtk
      ➔ Run 5/10 Selesai | Jarak: 197.9 KM | Waktu: 0.794 dtk
      ➔ Run 6/10 Selesai | Jarak: 223.78 KM | Waktu: 0.783 dtk
      ➔ Run 7/10 Selesai | Jarak: 197.31 KM | Waktu: 0.801 dtk
      ➔ Run 8/10 Selesai | Jarak: 198.43 KM | Waktu: 0.784 dtk
      ➔ Run 9/10 Selesai | Jarak: 224.87 KM | Waktu: 0.809 dtk
      ➔ Run 10/10 Selesai | Jarak: 200.38 KM | Waktu: 0.709 dtk
✅ HASIL TERBAIK KLASTER BARAT:
Total Kurir          : 2 Orang
Fitness Min (Jarak)  : 224.87 KM
Fitness Rata-Rata    : 305.56 KM
Standar Deviasi      : 57.48
Waktu Komputasi (Avg): 0.754 Detik
-----------------------